<a href="https://colab.research.google.com/github/YashVermaTech/neuromorphic-sda/blob/main/notebooks/01_data_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ Notebook 01 — Orbital Image → Event Stream Pipeline

**Neuromorphic Data Synthesis for Space Domain Awareness**  
*Author: Yash Verma · TU Darmstadt Aerospace Engineering*

---

This notebook demonstrates the complete data pipeline:
1. Generate / download a synthetic orbital image
2. Convert it to an asynchronous event stream using the **v2e** algorithm
3. Visualise the events as scatter plots, time surfaces, and event-rate histograms
4. Show a side-by-side comparison of original frame vs. event stream
5. Save and reload the event stream from disk

## 📦 Setup (Colab / local)

In [ ]:
# ── Install dependencies (only needed on Colab) ────────────────────────
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'snntorch>=0.7.0', 'opencv-python-headless',
                    'tqdm', 'scipy', 'plotly'], check=True)
    # Clone repo
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/YashVermaTech/neuromorphic-sda.git'], check=True)
    sys.path.insert(0, 'neuromorphic-sda/src')
else:
    # Local: assume repo root is one level up
    import os
    sys.path.insert(0, os.path.join('..', 'src'))

print('✅ Environment ready')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from data_pipeline.orbital_to_events import OrbitalToEvents, EventStream, EVENT_DTYPE
from utils.visualization import EventVisualizer, plot_event_stream

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('✅ Imports successful')

---
## 1️⃣  Generate Synthetic Orbital Imagery

We synthesise a realistic deep-space scene containing:
- A **static starfield** (background noise floor + Gaussian star PSFs)
- A **moving satellite** (bright point source traversing the FOV)
- Atmospheric **scintillation** (small per-frame brightness variations)

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────
SENSOR_W, SENSOR_H = 346, 260
N_FRAMES = 60          # 2 seconds @ 30 fps
FPS      = 30.0
SEED     = 42

rng = np.random.default_rng(SEED)

def make_starfield(h, w, n_stars=120, seed=0):
    """Create a static starfield background."""
    bg_rng = np.random.default_rng(seed)
    field = np.zeros((h, w), dtype=np.float32)
    xs = bg_rng.integers(5, w - 5, n_stars)
    ys = bg_rng.integers(5, h - 5, n_stars)
    brightness = bg_rng.exponential(30.0, n_stars).clip(5, 220)
    for x, y, b in zip(xs, ys, brightness):
        for dy in range(-3, 4):
            for dx in range(-3, 4):
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w:
                    field[ny, nx] += b * np.exp(-(dx**2 + dy**2) / 3.0)
    return np.clip(field, 0, 255).astype(np.float32)

# Static background (computed once)
starfield = make_starfield(SENSOR_H, SENSOR_W, seed=SEED)

# Satellite trajectory (linear, wraps around)
sat_x0, sat_y0 = 50.0, 130.0
sat_vx, sat_vy = 3.2, 0.8   # pixels/frame

frames = []
sat_positions = []
for i in range(N_FRAMES):
    # Scintillation: ±3 ADU per star per frame
    scintillation = rng.normal(0, 3, (SENSOR_H, SENSOR_W)).astype(np.float32)
    frame = (starfield + scintillation).clip(0, 255)
    
    # Add satellite PSF
    sx = (sat_x0 + i * sat_vx) % SENSOR_W
    sy = (sat_y0 + i * sat_vy) % SENSOR_H
    sat_positions.append((sx, sy))
    
    for dy in range(-4, 5):
        for dx in range(-4, 5):
            ny, nx = int(sy) + dy, int(sx) + dx
            if 0 <= ny < SENSOR_H and 0 <= nx < SENSOR_W:
                frame[ny, nx] += 180.0 * np.exp(-(dx**2 + dy**2) / 4.0)
    
    frames.append(np.clip(frame, 0, 255).astype(np.uint8))

print(f'✅  Generated {N_FRAMES} frames  |  resolution: {SENSOR_W}×{SENSOR_H}')
print(f'   Satellite start: ({sat_x0:.0f}, {sat_y0:.0f})  velocity: ({sat_vx:.1f}, {sat_vy:.1f}) px/frame')

In [ ]:
# Show a 3-panel sample of frames
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, idx, title in zip(axes, [0, 15, 45],
                           ['Frame 0  (t=0 ms)', 'Frame 15 (t=500 ms)', 'Frame 45 (t=1500 ms)']):
    ax.imshow(frames[idx], cmap='gray', vmin=0, vmax=255)
    sx, sy = sat_positions[idx]
    ax.plot(sx, sy, 'ro', markersize=10, markerfacecolor='none', markeredgewidth=2, label='Satellite')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
    ax.legend(fontsize=8)
fig.suptitle('Synthetic Orbital Imagery — Static Starfield + Moving Satellite', fontsize=13)
plt.tight_layout()
plt.show()

---
## 2️⃣  v2e Conversion: Frames → Event Stream

Each pixel independently fires an **ON** event when its log-luminance *increases* by ≥ `C_pos`,  
and an **OFF** event when it *decreases* by ≥ `C_neg`.

The satellite generates events at its edges; the static stars generate almost none.

In [ ]:
converter = OrbitalToEvents(
    sensor_width  = SENSOR_W,
    sensor_height = SENSOR_H,
    threshold_pos = 0.15,    # log-intensity contrast threshold
    threshold_neg = 0.15,
    threshold_sigma       = 0.03,   # per-pixel mismatch
    refractory_period_us  = 1.0,    # 1 μs dead time per pixel
    shot_noise_rate_hz    = 0.5,    # 0.5 noise events/pixel/second
    seed = SEED,
)

stream = converter.convert(frames, fps=FPS, show_progress=True)
print('\n', stream)
print(f'  ON  events : {stream.on_events.shape[0]:,}')
print(f'  OFF events : {stream.off_events.shape[0]:,}')
print(f'  Event rate : {stream.event_rate_hz/1e3:.1f} kHz')

---
## 3️⃣  Event Stream Visualisation

In [ ]:
# ── A) Scatter plot (full stream) ───────────────────────────────────────
viz = EventVisualizer(sensor_width=SENSOR_W, sensor_height=SENSOR_H, dpi=130)
fig = viz.scatter_plot(
    stream.events,
    title=f'Event Stream — {stream.num_events:,} events over {stream.duration_us/1e6:.1f} s',
    alpha=0.4,
    marker_size=1.0,
)
plt.show()

In [ ]:
# ── B) Time surface (last 100 ms of stream) ─────────────────────────────
t_end_us   = stream.duration_us
t_start_us = t_end_us - 100_000   # 100 ms window
window_events = stream.window(t_start_us, t_end_us)

fig = viz.time_surface(
    window_events,
    tau_us=20_000,
    title='Time Surface (last 100 ms)',
    colormap='inferno',
)
plt.show()

In [ ]:
# ── C) Event rate over time ─────────────────────────────────────────────
fig = viz.event_rate_plot(
    stream.events,
    bin_ms=10.0,
    title='Event Rate vs Time (10 ms bins)',
)
plt.show()

In [ ]:
# ── D) Before / after comparison ────────────────────────────────────────
# Use events from the middle 33 ms (one frame interval)
t_mid = stream.duration_us // 2
mid_events = stream.window(t_mid, t_mid + 33_333)

frame_idx = N_FRAMES // 2
fig = viz.comparison_plot(
    original_frame=frames[frame_idx],
    events=mid_events,
    title=f'Original Frame {frame_idx} vs Corresponding Event Window',
)
plt.show()
print(f'  Events in this 33 ms window: {len(mid_events):,}')

---
## 4️⃣  Event Frame Accumulation (SNN Input Format)

The SNN consumes **polarity frames** `(2, H, W)` accumulated over time windows.  
Channel 0 = ON events, Channel 1 = OFF events.

In [ ]:
window_ms = 50   # 50 ms accumulation window
window_us = window_ms * 1000

# Generate polarity frames for the entire stream
n_windows = stream.duration_us // window_us
polarity_frames = []
for i in range(n_windows):
    t0 = i * window_us
    t1 = t0 + window_us
    pf = stream.to_frame(t0, t1)   # (2, H, W)
    polarity_frames.append(pf)

print(f'Generated {len(polarity_frames)} polarity frames  |  shape: {polarity_frames[0].shape}')

# Visualise one window
pf = polarity_frames[len(polarity_frames) // 2]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(pf[0], cmap='Blues',  vmin=0, vmax=1);  axes[0].set_title('ON  events  (ch 0)')
axes[1].imshow(pf[1], cmap='Reds',   vmin=0, vmax=1);  axes[1].set_title('OFF events  (ch 1)')
combined = pf[0] - pf[1]
axes[2].imshow(combined, cmap='RdBu_r', vmin=-1, vmax=1); axes[2].set_title('Polarity diff (ON − OFF)')
for ax in axes:
    ax.axis('off')
fig.suptitle(f'Polarity Frame — {window_ms} ms window', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5️⃣  Polarity Statistics

In [ ]:
on_count  = stream.on_events.shape[0]
off_count = stream.off_events.shape[0]
total     = stream.num_events

# Pixel activation map
activation_map = np.zeros((SENSOR_H, SENSOR_W), dtype=np.float32)
if total > 0:
    np.add.at(activation_map, (stream.events['y'], stream.events['x']), 1.0)
    activation_map /= activation_map.max()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Polarity pie chart
axes[0].pie([on_count, off_count],
            labels=[f'ON\n{on_count:,}', f'OFF\n{off_count:,}'],
            colors=['#00BFFF', '#FF4500'],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title('Event Polarity Distribution', fontsize=12)

# Pixel activation heatmap
im = axes[1].imshow(activation_map, cmap='hot', origin='upper')
axes[1].set_title('Pixel Activation Map (total events per pixel)', fontsize=12)
plt.colorbar(im, ax=axes[1], label='Normalised count')

plt.tight_layout()
plt.show()

print(f'ON/OFF ratio     : {on_count/max(off_count,1):.3f}')
print(f'Active pixels    : {(activation_map > 0).sum():,} / {SENSOR_W*SENSOR_H:,}  ({100*(activation_map>0).mean():.1f}%)')
print(f'Mean event rate  : {stream.event_rate_hz/1e3:.2f} kHz')

---
## 6️⃣  Save & Reload Event Stream

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    save_path = Path(tmpdir) / 'orbital_stream.npz'
    stream.save_numpy(save_path)
    file_size_kb = os.path.getsize(save_path) / 1024
    
    # Reload
    loaded = EventStream.load_numpy(save_path)
    
    match = (loaded.num_events == stream.num_events and
             (loaded.events['t'] == stream.events['t']).all())
    
    print(f'Saved  : {save_path.name}  ({file_size_kb:.1f} KB)')
    print(f'Loaded : {loaded}')
    print(f'Roundtrip OK: {match} ✅' if match else '❌ Mismatch!')

---
## 7️⃣  Effect of Threshold on Event Count

In [ ]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
event_counts = []

for thresh in thresholds:
    c = OrbitalToEvents(sensor_width=SENSOR_W, sensor_height=SENSOR_H,
                        threshold_pos=thresh, threshold_neg=thresh,
                        shot_noise_rate_hz=0.0, seed=SEED)
    s = c.convert(frames[:20], fps=FPS, show_progress=False)
    event_counts.append(s.num_events)
    print(f'  Threshold {thresh:.2f} → {s.num_events:>8,} events')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(t) for t in thresholds], event_counts, color='steelblue', edgecolor='white')
ax.set_xlabel('Contrast Threshold (C)', fontsize=11)
ax.set_ylabel('Total Events (20 frames)', fontsize=11)
ax.set_title('Event Count vs Contrast Threshold', fontsize=13)
for bar, count in zip(ax.patches, event_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(event_counts)*0.01,
            f'{count:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## ✅ Summary

| Step | Result |
|------|--------|
| Frames generated | 60 frames @ 30 fps = 2 seconds |
| Event stream | Asynchronous, microsecond timestamps |
| Event format | Structured numpy: (t, x, y, p) |
| Visualisations | Scatter, time surface, space-time, rate histogram |
| SNN input | Polarity frames (2, H, W) per time window |
| I/O | Compressed .npz with full roundtrip |

➡️ **Next:** [Notebook 02 — GAN Cosmic Radiation Noise Modelling](./02_gan_noise_modeling.ipynb)